In [5]:
import time
import pyautogui as pyg

pyg.sleep(2)  # 給你 2 秒鐘切換到遊戲畫面
pyg.press("space")  # 啟動遊戲

start_time = time.time()

# ================= 座標調整區 =================
X_BASE = 550  # 基礎起跳距離
ROI_WIDTH = 150  # 加大區域寬度以對應 1000 分極速

Y_GROUND = 726  # 地面仙人掌底部
Y_MID = 690  # 低空鳥與中空仙人掌
Y_BIRD = 640  # 高空鳥高度
# =============================================

while True:
    elapsed = time.time() - start_time

    # 【1000 分極速動態補償 (大約 45 秒超高分區間)】
    if elapsed > 45:  # 過 1000 分：進入極限加速模式
        speed_offset = min(int((elapsed - 20) * 5.5), 340)
        jump_hold_time = 0.06  # 極速下微幅縮短滯空時間，利於連續落地
        scan_step = 2  # 提高掃描解析度，防止極速下漏抓
    elif elapsed > 35:  # 約 800 分階梯
        speed_offset = min(int((elapsed - 20) * 4.8), 290)
        jump_hold_time = 0.08
        scan_step = 3
    elif elapsed > 20:  # 約 450 分階梯
        speed_offset = min(int((elapsed - 20) * 3.3), 190)
        jump_hold_time = 0.08
        scan_step = 3
    else:  # 開局慢速區
        speed_offset = 0
        jump_hold_time = 0.08
        scan_step = 3

    current_x = X_BASE + speed_offset

    # 擷取恐龍前方區域
    roi = pyg.screenshot(region=(current_x, Y_BIRD - 10, ROI_WIDTH, 110))

    # 取出背景顏色 (自動適應白天/黑夜)
    p1 = roi.getpixel((0, 0))[0]
    p2 = roi.getpixel((0, 50))[0]
    bg_color = (p1 + p2) // 2

    has_ground = False
    has_bird = False

    # 區域內極速掃描
    for x in range(0, ROI_WIDTH, scan_step):
        c_ground = roi.getpixel((x, Y_GROUND - (Y_BIRD - 10)))[0]
        c_mid = roi.getpixel((x, Y_MID - (Y_BIRD - 10)))[0]
        c_bird = roi.getpixel((x, 10))[0]

        # 判定障礙物
        if abs(c_ground - bg_color) > 35 or abs(c_mid - bg_color) > 35:
            has_ground = True

        if elapsed > 23 and abs(c_bird - bg_color) > 35:
            has_bird = True

        if has_ground:
            break

    # 執行動作
    if has_ground:
        pyg.keyDown("space")
        pyg.sleep(jump_hold_time)  # 動態調整長壓時間
        pyg.keyUp("space")
    elif has_bird:
        pyg.keyDown("down")
        pyg.sleep(0.35)
        pyg.keyUp("down")

KeyboardInterrupt: 